# XGBoost – Klasifikace onemocnění srdce (Exercise 1)
Tento Jupyter sešit představuje kompletní vypracované řešení cvičení **XGBoost - exercise 1** ze 3. dne kurzu Machine Learning.

### Zadání cvičení:
1. Načíst data `heart_data_normalized.csv` pomocí knihovny Pandas.
2. Importovat potřebné třídy pro stavbu modelu XGBoost z knihovny `xgboost` a Scikit-learn.
3. Rozdělit dataset na trénovací a testovací sadu v poměru 70:30 se seedem `random_state=42`.
4. Vytvořit instanci modelu `XGBClassifier` s definovaným parametrem `objective` (`binary:logistic`).
5. Definovat slovník hyperparametrů `params` pro klíče: `max_depth`, `n_estimators`, `gamma`, `objective` a `learning_rate`.
6. Vytvořit objekt `GridSearchCV` pro optimalizaci s metrikou `scoring='precision'`.
7. Natrénovat model na trénovacích datech `X_train, y_train` s nejlepší sadou hyperparametrů.
8. Provést predikci na testovací sadě `X_test`.
9. Vygenerovat podrobný `classification_report` pro testovací sadu a matici záměn.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import xgboost as xgb

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
print('Knihovny úspěšně importovány.')

Knihovny úspěšně importovány.


In [2]:
# 1. Načtení dat heart_data_normalized.csv
data_path = '../data/MAL_downloadable materials_session 2/Day 3/heart_data_normalized.csv'
if not os.path.exists(data_path):
    data_path = 'heart_data_normalized.csv'

heart_data = pd.read_csv(data_path)
print('Rozměry datasetu:', heart_data.shape)
print('Rozdělení cílové proměnné (ahd_yes):')
print(heart_data['ahd_yes'].value_counts())
heart_data.head()

Rozměry datasetu: (303, 18)
Rozdělení cílové proměnné (ahd_yes):
0    164
1    139
Name: ahd_yes, dtype: int64


In [3]:
# 2. & 3. Příprava matice X a cílového vektoru y + Split 70:30 (random_state=42)
X = heart_data.drop('ahd_yes', axis=1)
y = heart_data['ahd_yes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'Trénovací sada: {X_train.shape}, Testovací sada: {X_test.shape}')

Trénovací sada: (212, 17), Testovací sada: (91, 17)


In [4]:
# 4., 5., 6. & 7. XGBClassifier + Slovník params + GridSearchCV (scoring='precision')
xgb_classifier = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

params = {
    'max_depth': [3, 5, 7],
    'n_estimators': [50, 100, 150],
    'gamma': [0, 0.5, 1.0],
    'objective': ['binary:logistic'],
    'learning_rate': [0.05, 0.1, 0.2]
}

grid_search = GridSearchCV(
    estimator=xgb_classifier,
    param_grid=params,
    scoring='precision',
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_xgb = grid_search.best_estimator_
print('GridSearchCV dokončen.')
print('Nejlepší hyperparametry:', grid_search.best_params_)
print(f"Nejvyšší validační Precision (CV=5): {grid_search.best_score_*100:.2f} %")

GridSearchCV dokončen.
Nejlepší hyperparametry: {'gamma': 1.0, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 50, 'objective': 'binary:logistic'}
Nejvyšší validační Precision (CV=5): 82.63 %


In [5]:
# 8. & 9. Predikce na testovací sadě a classification_report
y_pred = best_xgb.predict(X_test)

print('=== CLASSIFICATION REPORT (Testovací sada 30 %) ===')
print(classification_report(y_test, y_pred))

print('Matice záměn (Confusion Matrix):')
cm = confusion_matrix(y_test, y_pred)
print(cm)

=== CLASSIFICATION REPORT (Testovací sada 30 %) ===
              precision    recall  f1-score   support

           0       0.83      0.83      0.83        48
           1       0.81      0.81      0.81        43

    accuracy                           0.82        91
   macro avg       0.82      0.82      0.82        91
weighted avg       0.82      0.82      0.82        91

Matice záměn (Confusion Matrix):
[[40  8]
 [ 8 35]]


In [6]:
# Vizualizace 1: Matice záměn (Confusion Matrix)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['Zdravý (0)', 'Nemoc srdce (1)'],
            yticklabels=['Zdravý (0)', 'Nemoc srdce (1)'])
plt.title('Matice záměn: XGBoost na testovací sadě', fontsize=13)
plt.xlabel('Predikovaná třída')
plt.ylabel('Skutečná třída')
plt.tight_layout()
plt.show()

In [7]:
# Vizualizace 2: Důležitost příznaků (XGBoost Feature Importance – Gain)
feat_importances = pd.Series(best_xgb.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feat_importances.plot(kind='barh', color='#f97316')
plt.title('Důležitost příznaků (XGBoost Importance – Gain)', fontsize=13)
plt.xlabel('F-Score / Gain')
plt.tight_layout()
plt.show()

### Závěrečné vyhodnocení a srovnání s Random Forestem:
- **Výsledky XGBoost modelu na testu:**
  - **Accuracy:** 82.42 %
  - **Precision (třída 1 – nemoc srdce):** 81.40 %
  - **Recall (třída 1 – nemoc srdce):** 81.40 %
  - **F1-Score:** 0.8140
- **Srovnání s Random Forestem ze cvičení 1:**
  - Random Forest dosáhl Precision **85.71 %** a Recall **83.72 %**.
  - Na takto malém datasetu (pouze 212 trénovacích pacientů a 17 příznaků) je sekvenční Boosting náchylnější k zachycení šumu než paralelní Bagging, který těží ze silné redukce rozptylu průměrováním.
  - Nastavení regularizace `gamma = 1.0` v XGBoostu pomohlo zabránit větším přeučením.